# Base policy analysis

Per-step eval traces of one wandb group's runs (the parquet `scripts/eval.py --run ...` uploads to the
run's files, e.g. with `configs/eval/one_leg_phases.yaml`, which sets `task.cfg.furniture.track_phases`).

Cells: 1 imports, 2 variables, 3 download, 4 preprocessing, then one plot or table per cell.
With more than one run in the group, the plots show the mean over runs with a 95% confidence interval
(Student t over the per-run rates).

In [ ]:
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import to_rgb
from matplotlib.patches import PathPatch, Rectangle
from matplotlib.path import Path as MplPath
import pandas as pd
import wandb
from scipy import stats

# the repo root, so the phase ids come from the one_leg overlay itself
REPO = Path.cwd().resolve()
while not (REPO / "my_project").is_dir():
    if REPO.parent == REPO:
        raise RuntimeError("run this notebook from inside the hybrid_demos repo")
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
from my_project.overlays import PHASE_NAMES  # noqa: E402

In [ ]:
# the runs: every run in this wandb group
ENTITY = "hur"
PROJECT = "hybrid_demo_furniture"
GROUP = "physic_aligned_evalHappy"

# which eval of each run: the eval config's stem (the parquet is <stem>.parquet in the run's
# eval/<stem>_<timestamp>/ or videos/<stem>_<timestamp>/ folder); a run with several such evals
# uses its latest
EVAL_STEM = "one_leg_phases"

# where the parquets are downloaded to (git-ignored); a file already there is not downloaded again
CACHE_DIR = REPO / "data_analysis" / "cache"

# the per-step phase column (the env's eval_state channel 'phase/id')
PHASE_COLUMN = "eval_state/phase/id"

# True: timeouts are counted as failures (one 'Failure' node); False: a separate 'Timeout' node
MERGE_TIMEOUT_INTO_FAILURE = True

# the confidence level of the across-run intervals
CONFIDENCE = 0.95

# the figures: size in inches, resolution, font
FIG_SIZE = (12, 5)
DPI = 200
FONT_FAMILY = "DejaVu Sans"
FONT_SIZE = 11

# Sankey colours: the first bar (Corner) START_COLOUR, Success END_COLOUR, the bars between linearly
# interpolated; the drops and their outcome bars FAILURE_COLOUR / TIMEOUT_COLOUR
START_COLOUR = "#f2c218"  # yellow
END_COLOUR = "#2ca02c"  # green
FAILURE_COLOUR = "#d62728"
TIMEOUT_COLOUR = "#8c8c8c"
LINK_ALPHA = 0.35
# the Sankey boxes' width (the columns are 1 apart); the labels are written inside them
BOX_WIDTH = 0.45

# where each figure is saved (its name + the extension, e.g. ".png" or ".svg"); None: shown only
SAVE_DIR = None  # e.g. REPO / "data_analysis" / "figures" / GROUP
SAVE_FORMATS = [".png", ".svg"]


In [20]:
api = wandb.Api()
runs = list(api.runs(f"{ENTITY}/{PROJECT}", filters={"group": GROUP}))
if not runs:
    raise LookupError(f"no runs in {ENTITY}/{PROJECT} group {GROUP!r}")

trace_name = re.compile(rf"^(eval|videos)/{re.escape(EVAL_STEM)}_(\d{{8}}-\d{{6}})/{re.escape(EVAL_STEM)}\.parquet$")

frames, missing = [], []
for run in runs:
    found = [(m.group(2), f) for f in run.files() if (m := trace_name.match(f.name))]
    if not found:
        missing.append(run.name)
        continue
    stamp, handle = max(found, key=lambda x: x[0])  # the latest eval
    root = CACHE_DIR / ENTITY / PROJECT / GROUP / run.id
    path = root / handle.name
    if not path.exists():
        handle.download(root=str(root), replace=True)
    frame = pd.read_parquet(path)
    frame.insert(0, "run", run.name)
    frame.insert(1, "eval", f"{EVAL_STEM}_{stamp}")
    frames.append(frame)
    print(f"{run.name} ({run.id}): {handle.name}, {len(frame)} rows")

if missing:
    print(f"no {EVAL_STEM} eval in {len(missing)} of {len(runs)} runs, left out: {missing}")
if not frames:
    raise FileNotFoundError(f"none of the {len(runs)} runs in group {GROUP!r} has a {EVAL_STEM} eval trace")
steps = pd.concat(frames, ignore_index=True)
print(f"{steps['run'].nunique()} runs, {len(steps)} steps")

physic_aligned_evalHappy_a0 (1vs2w5ji): eval/one_leg_phases_20261021-114409/one_leg_phases.parquet, 407650 rows
1 runs, 407650 steps


In [21]:
# phase ids (the env's furniture.track_phases): 0-3 the task phases, 4-6 how the episode ended (its last step)
TASK_PHASES = [name.capitalize() for name in PHASE_NAMES[:4]]  # Corner, Grasp, Insert, Screw
SUCCESS, FAILURE, TIMEOUT = 4, 5, 6

if PHASE_COLUMN not in steps:
    raise KeyError(
        f"the traces have no {PHASE_COLUMN!r} column: eval with task.cfg.furniture.track_phases: true "
        f"(configs/eval/one_leg_phases.yaml)"
    )
if steps[PHASE_COLUMN].isna().any():
    raise ValueError(f"{PHASE_COLUMN} has NaN rows: some rounds were evaluated without phase tracking")

# one episode per (run, round, env): each env's first episode of the round
steps = steps.sort_values(["run", "round", "env", "step"])
steps["phase"] = steps[PHASE_COLUMN].round().astype(int)
grouped = steps.groupby(["run", "round", "env"], sort=False)["phase"]
task = lambda p: p[p < SUCCESS]  # noqa: E731  the task-phase steps (the ending step is 4-6)
episodes = pd.DataFrame({
    "last": grouped.last(),
    # the phase id is the CURRENT phase: the one the episode ended in was not completed
    "ended_in": grouped.apply(lambda p: task(p).iloc[-1] if len(task(p)) else 0),
    "went_back": grouped.apply(lambda p: bool((np.diff(task(p).to_numpy()) < 0).any())),
})
if (episodes["last"] < SUCCESS).any():
    bad = episodes[episodes["last"] < SUCCESS]
    raise ValueError(f"{len(bad)} episodes have no ending phase (4-6) on their last step:\n{bad.head()}")
if episodes["went_back"].any():
    print(f"note: {int(episodes['went_back'].sum())} episodes went back to an earlier phase; "
          "each is counted by the phase it ended in")

# task phases completed: those before the one it ended in; a success completed all of them
episodes["completed"] = np.where(episodes["last"] == SUCCESS, len(TASK_PHASES), episodes["ended_in"])
outcome_names = {SUCCESS: "Success", FAILURE: "Failure", TIMEOUT: "Failure" if MERGE_TIMEOUT_INTO_FAILURE else "Timeout"}
episodes["outcome"] = episodes["last"].map(outcome_names)
episodes = episodes.reset_index()

# per run: episodes completing each task phase, ending in each outcome, and the Sankey's drops:
# '<phase>-><outcome>' = ended in <phase> (entered it, did not complete it) as <outcome>
OUTCOMES = ["Success", "Failure"] + ([] if MERGE_TIMEOUT_INTO_FAILURE else ["Timeout"])


def run_rates(ep):
    row = {"episodes": len(ep)}
    for i, name in enumerate(TASK_PHASES):
        row[name] = (ep["completed"] > i).sum()
    for name in OUTCOMES:
        row[name] = (ep["outcome"] == name).sum()
    for i, name in enumerate(TASK_PHASES):
        stopped = ep[ep["completed"] == i]
        for out in OUTCOMES:
            row[f"{name}->{out}"] = (stopped["outcome"] == out).sum()
    return pd.Series(row)


counts = episodes.groupby("run").apply(run_rates)
rates = counts.drop(columns="episodes").div(counts["episodes"], axis=0)
n_runs = len(rates)


def interval(column):
    """The mean over runs and its CONFIDENCE half-width (Student t; 0 with one run)."""
    x = rates[column].to_numpy(float)
    if n_runs < 2:
        return x.mean(), 0.0
    return x.mean(), stats.t.ppf(0.5 + CONFIDENCE / 2, n_runs - 1) * x.std(ddof=1) / np.sqrt(n_runs)


summary = pd.DataFrame(
    [(c, *interval(c), counts[c].sum(), counts["episodes"].sum()) for c in TASK_PHASES + OUTCOMES],
    columns=["node", "mean", "ci_half_width", "episodes_pooled", "of_pooled"],
).set_index("node")
display(counts)
display(summary.style.format({"mean": "{:.1%}", "ci_half_width": "{:.1%}"}))

,episodes,Corner,Grasp,Insert,Screw,Success,Failure,Corner->Success,Corner->Failure,Grasp->Success,Grasp->Failure,Insert->Success,Insert->Failure,Screw->Success,Screw->Failure
run,,,,,,,,,,,,,,,
physic_aligned_evalHappy_a0,512,473,460,365,280,280,232,0,39,0,13,0,95,0,85


,mean,ci_half_width,episodes_pooled,of_pooled
node,,,,
Corner,92.4%,0.0%,473,512
Grasp,89.8%,0.0%,460,512
Insert,71.3%,0.0%,365,512
Screw,54.7%,0.0%,280,512
Success,54.7%,0.0%,280,512
Failure,45.3%,0.0%,232,512


In [ ]:
# phase progression: Corner holds every episode (each starts there); each later phase's bar holds the
# episodes that entered it (completed the one before). From a bar, those that completed the phase go on
# to the next, and those that did not go red to how they ended. A label gives the phase's COMPLETED
# count, which is the width of its forward link (into Success, for Screw).
# Heights are the mean fraction of episodes over the group's runs, so every bar's in- and outflow match.
chain = TASK_PHASES + ["Success"]  # yellow -> green
drops = [o for o in OUTCOMES if o != "Success"]  # Failure (and Timeout)
start, end = np.array(to_rgb(START_COLOUR)), np.array(to_rgb(END_COLOUR))
colour = {n: tuple(start + (end - start) * k / (len(chain) - 1)) for k, n in enumerate(chain)}
colour.update({"Failure": to_rgb(FAILURE_COLOUR), "Timeout": to_rgb(TIMEOUT_COLOUR)})

mean = lambda column: interval(column)[0]  # noqa: E731
completed = {n: mean(n) for n in TASK_PHASES}
entered = {n: 1.0 if i == 0 else completed[TASK_PHASES[i - 1]] for i, n in enumerate(TASK_PHASES)}
dropped = {(n, o): mean(f"{n}->{o}") for n in TASK_PHASES for o in drops}

# every bar's outflow must be the episodes that entered it, or the picture lies
for n in TASK_PHASES:
    out = completed[n] + sum(dropped[n, o] for o in drops)
    if not np.isclose(out, entered[n]):
        raise AssertionError(f"{n}: entered {entered[n]:.3f}, outflow {out:.3f}")

# layout: one column per phase, the outcomes in the last; bars hang from y = 1 (1 = every episode);
# each bar's forward flow on top, its drops under it
W, GAP = BOX_WIDTH, 0.06  # box width (columns are 1 apart), gap between the outcome boxes
x = {n: float(i) for i, n in enumerate(TASK_PHASES)}
x.update({o: float(len(TASK_PHASES)) for o in OUTCOMES})
span = {n: (1.0 - entered[n], 1.0) for n in TASK_PHASES}  # (bottom, top)
top = 1.0
for o in OUTCOMES:  # Success, then Failure (then Timeout) under it
    h = mean(o)
    span[o] = (top - h, top)
    top -= h + GAP


def band(ax, x0, y0, x1, y1, h, c):
    """A link of height h from (x0, top y0) to (x1, top y1): two cubic curves, flat at both ends."""
    if h <= 0:
        return
    xm = (x0 + x1) / 2
    verts = [(x0, y0), (xm, y0), (xm, y1), (x1, y1), (x1, y1 - h), (xm, y1 - h), (xm, y0 - h), (x0, y0 - h), (x0, y0)]
    codes = [MplPath.MOVETO] + [MplPath.CURVE4] * 3 + [MplPath.LINETO] + [MplPath.CURVE4] * 3 + [MplPath.CLOSEPOLY]
    ax.add_patch(PathPatch(MplPath(verts, codes), facecolor=c, edgecolor="none", alpha=LINK_ALPHA))


total = int(counts["episodes"].sum())


def stat(name):
    pooled = int(summary.loc[name, "episodes_pooled"])
    if n_runs == 1:
        return f"{pooled}/{total}"
    m, half = summary.loc[name, ["mean", "ci_half_width"]]
    return f"{m:.0%} ± {half:.0%}\n({pooled}/{total})"


def ink(c):
    """Black or white text, whichever reads better on the box colour c."""
    r, g, b = c
    return "black" if 0.299 * r + 0.587 * g + 0.114 * b > 0.5 else "white"


with plt.rc_context({"font.family": FONT_FAMILY, "font.size": FONT_SIZE}):
    fig, ax = plt.subplots(figsize=FIG_SIZE, dpi=DPI)
    for i, n in enumerate(TASK_PHASES):  # forward: the top of each bar to the top of the next
        nxt = TASK_PHASES[i + 1] if i + 1 < len(TASK_PHASES) else "Success"
        band(ax, x[n] + W / 2, span[n][1], x[nxt] - W / 2, span[nxt][1], completed[n], colour[nxt])
    # drops: under the forward flow; an outcome bar fills from the top, last phase first, so they do not cross
    slot = {}
    for o in drops:
        y_in = span[o][1]
        for n in reversed(TASK_PHASES):
            slot[n, o] = y_in
            y_in -= dropped[n, o]
    for n in TASK_PHASES:
        y = span[n][1] - completed[n]
        for o in drops:
            band(ax, x[n] + W / 2, y, x[o] - W / 2, slot[n, o], dropped[n, o], colour[o])
            y -= dropped[n, o]

    for n in TASK_PHASES + OUTCOMES:
        b, t = span[n]
        ax.add_patch(Rectangle((x[n] - W / 2, b), W, t - b, facecolor=colour[n], edgecolor="none"))
    for n in TASK_PHASES + OUTCOMES:  # inside each box: its name and count (a phase's: completed)
        b, t = span[n]
        text = f"{n}\ncompleted\n{stat(n)}" if n in TASK_PHASES else f"{n}\n{stat(n)}"
        ax.text(x[n], (b + t) / 2, text, ha="center", va="center", color=ink(colour[n]))

    ci_note = "" if n_runs == 1 else f", mean ± {CONFIDENCE:.0%} CI over {n_runs} runs"
    ax.set_title(f"{GROUP}: phase progression ({EVAL_STEM}{ci_note})", pad=FONT_SIZE)
    ax.set_xlim(-W / 2 - 0.02, len(TASK_PHASES) + W / 2 + 0.02)
    ax.set_ylim(top + GAP - 0.02, 1.0)
    ax.axis("off")
    if SAVE_DIR is not None:
        Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)
        for ext in SAVE_FORMATS:
            fig.savefig(Path(SAVE_DIR) / f"phase_sankey{ext}", dpi=DPI, bbox_inches="tight")
    plt.show()